In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append('../scripts')

In [ ]:
import numpy as np
import scanpy as sc
import os
DATA_ROOT = os.environ.get("DATA_ROOT", ".")
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from cellina import CellinaModel
from utils import set_seed
from train_loo import preprocess_crc, preprocess_spatial_features

In [ ]:
plt.rcParams['font.family'] = 'monospace'

In [ ]:
import cellina
cellina.__version__

In [ ]:
base_path = "cellina-reproducibility"

In [ ]:
def plot_spatial(
    adata,
    color,
    fig_save_path='../figures',
    spot_size=200,
    title=None,
    cmap='magma',
    vmin=0,
    vmax=10,
    figsize=(10, 10),
    save=True,
    show=True,
    fname=None,
):
    title = title if title is not None else str(color)

    fig, ax = plt.subplots(figsize=figsize)

    sc.pl.spatial(
        adata,
        color=color,
        show=False,
        ax=ax,
        spot_size=spot_size,
        title='',
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(title, fontsize=22, fontweight='bold', pad=12, loc='left')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)

    # ── strip scanpy's colorbar axes and rebuild ──────────────────────────────
    for a in [a for a in fig.axes if a is not ax]:
        a.remove()

    mappable = ax.collections[-1] if ax.collections else None
    if mappable is not None:
        # [left, bottom, width, height] in figure coords — tweak width for thickness
        cbar_ax = fig.add_axes([0.88, 0.15, 0.04, 0.65])
        cbar = fig.colorbar(mappable, cax=cbar_ax)
        cbar.ax.tick_params(labelsize=22, length=8, width=2)
        cbar.set_label(str(color), fontsize=22, labelpad=12)

    # ── rasterize ─────────────────────────────────────────────────────────────
    for coll in ax.collections:
        coll.set_rasterized(True)
    for img in ax.images:
        img.set_rasterized(True)

    if show:
        plt.show()

    if save:
        if fname is None:
            fname = f"{fig_save_path}/{color}_spatial.svg"
        else:
            fname = f"{fig_save_path}/{fname}"
        fig.savefig(fname, dpi=300, bbox_inches='tight', transparent=True)
        print(f"Saved → {fname}")

    return fig, ax

In [ ]:

spot_size = 200
slide_id = 210

# Get dataset

In [ ]:
set_seed(0)

In [ ]:
data_path = os.path.join(DATA_ROOT, "datasets/crc/raw_zenodo/crc_210.h5ad")
adata = sc.read(data_path)
adata.obs_names_make_unique()

In [ ]:
adata = preprocess_crc(adata)

In [ ]:
adata

In [ ]:
labels_key = 'coarse_type'
domains_key = 'typ_clean'
batch_key = 'sid'

In [ ]:
sc.pl.spatial(adata, color=[domains_key, labels_key], spot_size=100)

## Data splits

In [ ]:
split = "ood"

# Get holdout indices
if split == "random":
    fraction = 0.1
    n_cells = adata.n_obs
    n_holdout = int(n_cells * fraction)

    # Randomly choose cells
    test_idx = np.random.choice(n_cells, n_holdout, replace=False)

elif split == "ood":
    holdout_ct = "Fibroblast"
    is_tumor_region  = adata.obs[domains_key].str.contains("CRC", regex=True)
    is_holdout_ct = adata.obs[labels_key] == holdout_ct

    # Combine for test set
    test_mask = (is_tumor_region) & (is_holdout_ct)
    test_idx = np.where(test_mask)[0]
else:
    raise ValueError(f"Unknown split: {split}")

# Get train/val indices
all_idx = np.arange(adata.n_obs)
trainval_idx = np.setdiff1d(all_idx, test_idx)

In [ ]:
preprocess_spatial_features(adata, step_size_px=0.12028, n_neighbors=200, test_indices=test_idx)

In [ ]:
# Set 'is_holdout' to False by default, then True for selected cells
adata.obs['is_holdout'] = False
adata.obs.iloc[test_idx, adata.obs.columns.get_loc('is_holdout')] = True

In [ ]:
validation_size = 0.1
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=validation_size,
    random_state=0,
    shuffle=True,
)

# Train

In [ ]:
from scvi.train._callbacks import SaveCheckpoint, EarlyStopping

batch_size = 4096
model_args = {
    'adata': adata,
    'n_latent': 64,
    'n_layers': 3,
    'use_observed_lib_size': True,
    'condition_on_intrinsic': False,
    'gene_likelihood': 'nb',
    'classifier_lambda': 1.,
    'discriminator_lambda': 1.,
    }
train_args = {'max_epochs': 100,
              'batch_size': batch_size,
              'check_val_every_n_epoch': 1,
              'early_stopping': True,
              'devices': [0],
              'enable_checkpointing':True,
              'datasplitter_kwargs': {
                  "external_indexing": [train_idx, val_idx, test_idx],
                  },
              'callbacks': [
                  SaveCheckpoint(
                      monitor='vae_loss_validation',
                      dirpath=f"{base_path}/trained/counterfactual_demo/{holdout_ct}",
                      load_best_on_end=True,
                      ),
                  EarlyStopping(
                      monitor="vae_loss_validation",
                      patience=5,
                      mode="min",
                    ),
                ],
    }

plan_kwargs = {
    'lr': 1e-3,
    'normalize_losses': True
    }

In [ ]:
CellinaModel.setup_anndata(adata,
                           batch_key=batch_key,
                           labels_key=labels_key, 
                           domains_key=domains_key, 
                           spatial_obsm_key="spatial_x",
                           layer='counts')

In [ ]:
model = CellinaModel(
    **model_args,
)
model.train(
    **train_args,
    plan_kwargs=plan_kwargs
)

model.save(f"trained/crc_210_{split}", overwrite=True)

# Analysis

In [ ]:
checkpoint_name = os.listdir(f"{base_path}/trained/counterfactual_demo/{holdout_ct}")[0]
model = CellinaModel.load(
    f"{base_path}/trained/counterfactual_demo/{holdout_ct}/{checkpoint_name}",
    adata=adata,
)

## Latent visualization

In [ ]:
adata.obsm['cellina_basal'] = model.get_latent_representation(latent_key='z', batch_size=batch_size)
adata.obsm['cellina_spatial'] = model.get_latent_representation(latent_key='s', batch_size=batch_size)
adata.obsm['cellina_joint'] = model.get_latent_representation(batch_size=batch_size)

In [ ]:
x = 0.1  # fraction of cells to keep (e.g., 20%)

n_cells = adata.n_obs
n_subsample = int(n_cells * x)

# Randomly choose cell indices
np.random.seed(42)  # for reproducibility
subsample_idx = np.random.choice(n_cells, n_subsample, replace=False)

# Create the subsampled AnnData
adata_sub = adata[subsample_idx].copy()

In [ ]:
sc.pp.neighbors(adata_sub, use_rep='cellina_basal')
sc.tl.umap(adata_sub)

In [ ]:
sc.pl.umap(adata_sub, color=[domains_key, labels_key, 'is_holdout'], wspace=0.4)

In [ ]:
sc.pp.neighbors(adata_sub, use_rep='cellina_spatial')
sc.tl.umap(adata_sub)

In [ ]:
sc.pl.umap(adata_sub, color=[domains_key, labels_key, 'is_holdout'], wspace=0.4)

# Counterfactual plot

In [ ]:
adata_exp = adata[(adata.obs[labels_key] == holdout_ct) & (adata.obs[domains_key].isin(['REF', 'CRC']))]

In [ ]:
is_tumor_region = adata_exp.obs[domains_key].astype(str).str.contains('CRC', regex=True)
mask_target = is_tumor_region & (adata_exp.obs[labels_key].astype(str) == holdout_ct)
idx_target = np.where(mask_target.values)[0]
#mask_control = (~adata_exp.obs['is_holdout']) & (adata_exp.obs[labels_key] == holdout_ct)
mask_control = (adata_exp.obs[domains_key].astype(str).str.contains('REF')) & (adata_exp.obs[labels_key] == holdout_ct)
idx_control = np.where(mask_control.values)[0]

In [ ]:
args_gex = {
    "adata": adata_exp,
    "indices": idx_control,
    "neighbour_indices": idx_target,
    "batch_size": batch_size,
    "seed": 0,
    "library_size": 1e4
}
cf_counts = model.get_counterfactual_expression(**args_gex)

args_latents = args_gex.copy() # Default gets 'shifted' latents, can set 'z' or 's' here
args_latents['latent_key'] = 's'
del args_latents['library_size']
cf_latents = model.get_counterfactual_latents(**args_latents)

In [ ]:
adata_exp.uns['counterfactual_x'] = cf_counts
adata_exp.uns['counterfactual_latents'] = cf_latents

In [ ]:
from scipy.stats import pearsonr, spearmanr
from counterfactual_analysis import safe_log2_fold_change

def _normalize_counts(counts, counts_per_k=1e4, eps=1e-8):
    return counts / (counts.sum(axis=1, keepdims=True) + eps) * counts_per_k

def compute_correlations(control, target, counterfactual, normalize_counts=True, deg=200):
    if normalize_counts:
        control = _normalize_counts(control)
        target = _normalize_counts(target)
        counterfactual = _normalize_counts(counterfactual)

    mean_control = np.nanmean(control, axis=0)
    mean_target = np.nanmean(target, axis=0)
    mean_cf = np.nanmean(counterfactual, axis=0)

    # compute log2 fold changes
    gt_vec = safe_log2_fold_change(mean_target, mean_control)
    cf_vec = safe_log2_fold_change(mean_cf, mean_control)

    deg_scores = np.abs(gt_vec)
    top_features = np.argsort(-deg_scores)[:deg]
    pear, _ = pearsonr(gt_vec[top_features], cf_vec[top_features])
    spear, _ = spearmanr(gt_vec[top_features], cf_vec[top_features])
    
    # Plot scatterplot of gt vs cf log fold changes - highlight top features in a different color
    import matplotlib.pyplot as plt
    plt.scatter(gt_vec, cf_vec)
    plt.xlabel('Ground Truth Log2 Fold Change')
    plt.ylabel('Counterfactual Log2 Fold Change')
    plt.title('GT vs CF Log2 Fold Change')

    # highlight genes with highest absolute log fold change in the ground truth by adding name to plot
    for i in top_features[:5]:
        plt.annotate(adata.var_names[i], (gt_vec[i], cf_vec[i]), fontsize=8)

    # highlight top features in a different color
    plt.scatter(gt_vec[top_features], cf_vec[top_features], color='red')
    plt.show()
    
    return float(pear), float(spear), top_features

In [ ]:
deg = 50

control = adata_exp.layers['counts'].todense()[mask_control]
control = np.asarray(control)

target = adata_exp.layers['counts'].todense()[mask_target]
target = np.asarray(target)

counterfactual = adata_exp.uns['counterfactual_x']
pear_global, spear_global, top_genes = compute_correlations(control, target, counterfactual, deg=deg)

In [ ]:
pear_global, spear_global

In [ ]:
# get gene names for top features
gene_names = adata_exp.var_names.values
top_gene_names = gene_names[top_genes]
top_gene_names

In [ ]:
gene = "IGF2"
#gene = "MS4A13"
gene_idx = adata.var_names.get_loc(gene)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))  # width x height in inches

sc.pl.spatial(
    adata,
    color=domains_key,
    show=True,
    ax=ax,
    spot_size=spot_size,
    title=f"Spatial domains"
)
fig.savefig("../figures/crc_210_domains.svg", bbox_inches='tight', dpi=200)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))  # width x height in inches

sc.pl.spatial(
    adata,
    color=labels_key,
    show=True,
    ax=ax,
    spot_size=spot_size,
    title=f"Cell types"
)
fig.savefig("../figures/crc_210_celltypes.svg", bbox_inches='tight')

In [ ]:
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
gene_key = f"{gene}_masked"
adata.obs[gene_key] = 0.0  # default

mask = (adata.obs[labels_key] == holdout_ct) & (adata.obs[domains_key].str.contains("CRC", regex=True))

# pull expression from X (or layer if needed)
adata.obs.loc[mask, gene_key] = adata[mask, gene].X.toarray().flatten()

In [ ]:
plot_spatial(
    adata[adata.obs['CenterY_global_px']>70000],
    #color=f"{gene}",
    color=gene_key,
    show=True,
    spot_size=spot_size*2,
    title=f"{gene} expression",
    cmap='magma',
    vmin=0,
    vmax=8,
    fig_save_path=f"../figures"
)


In [ ]:
plot_spatial(
    adata[adata.obs['CenterY_global_px']>70000],
    #color=f"{gene}",
    color=domains_key,
    show=True,
    spot_size=spot_size,
    title=f"{gene} expression",
    cmap='magma',
    vmin=0,
    vmax=10,
    fig_save_path=f"../figures"
)


In [ ]:
#ref_mask = (adata.obs[domains_key] != 'CRC') & (adata.obs[labels_key] == holdout_ct).values
ref_mask = (adata.obs[domains_key] == 'REF') & (adata.obs[labels_key] == holdout_ct).values

In [ ]:
# 1. Subset domains
adata_vis = adata[adata.obs[domains_key].isin(['REF', 'TVA'])].copy()

# 2. Create masked gene expression
gene_key = f"{gene}_masked"

adata_vis.obs[gene_key] = 0.0  # default

mask = adata_vis.obs[labels_key] == holdout_ct

# pull expression from X (or layer if needed)
adata_vis.obs.loc[mask, gene_key] = adata_vis[mask, gene].X.toarray().flatten()

In [ ]:
plot_spatial(
    adata_vis,
    #color=f"{gene}",
    color=gene_key,
    show=True,
    spot_size=spot_size*2,
    title=f"{gene} expression",
    fname=f"{slide_id}_{gene}_control2.svg",
    cmap='magma',
    vmin=0,
    vmax=8,
    fig_save_path=f"../figures"
)


In [ ]:
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
# Replace cells X which are Fib and REF with counterfactual
X_obs = adata.X.todense().copy()
X_obs[ref_mask, :] = adata_exp.uns['counterfactual_x']
adata.X = X_obs

In [ ]:
sc.pp.log1p(adata)

In [ ]:
gene_key = f"{gene}_masked"
adata.obs[gene_key] = 0.0  # default

mask = (adata.obs[labels_key] == holdout_ct)  | (adata.obs[domains_key].str.contains("CRC", regex=True))

# pull expression from X (or layer if needed)
adata.obs.loc[mask, gene_key] = adata[mask, gene].X.toarray().flatten()

In [ ]:
# 1. Subset domains
adata_vis = adata[adata.obs[domains_key].isin(['REF', 'TVA'])].copy()

# 2. Create masked gene expression
gene_key = f"{gene}_masked"

adata_vis.obs[gene_key] = 0.0  # default

mask = adata_vis.obs[labels_key] == holdout_ct

# pull expression from X (or layer if needed)
adata_vis.obs.loc[mask, gene_key] = adata_vis[mask, gene].X.toarray().flatten()

In [ ]:
plot_spatial(
    adata_vis,
    #color=f"{gene}",
    color=gene_key,
    show=True,
    spot_size=spot_size*2,
    title=f"{gene} expression",
    fname=f"crc_210_{gene}_perturbed.svg",
    cmap='magma',
    vmin=0,
    vmax=8,
    fig_save_path=f"../figures"
)
